## LLaDA-8B-Instruct · 4-bit inference · one GPU

This notebook teaches text generation with a masked diffusion language model (dLLM).
It includes installation, hardware checks, a tiny quantization test, model loading,
a readable denoising sampler, a generation trace, and an art-related text example.



### Before starting
1. Import this `.ipynb` into a new Kaggle Notebook (File → Import Notebook).
2. Set **Accelerator → GPU T4 x2** if offered. The default uses **only GPU 0**;
   two 16 GB GPUs do not automatically become one 32 GB GPU. A P100 is an alternative
   when its PyTorch/CUDA build passes the checks below. Do not select TPU.
3. Enable **Internet**. If GPU/Internet controls are unavailable, complete any account
   verification Kaggle requests and check remaining GPU quota/availability.
4. Start with a fresh session and run cells in order. Install before importing Transformers.
   If the installer says to restart, restart the kernel/session and run from the top.
5. No dataset or paid inference API is needed. The checkpoint is public; normally no
   Hugging Face token is required. An optional Kaggle secret is supported below.

### Resource expectations
- One mostly empty **16 GB NVIDIA GPU**, about **16 GB or more system RAM** recommended.
- Allow **at least 20 GB free cache disk**. The original approximately 16 GB weights are
  downloaded even though they are quantized into 4-bit for GPU use.
- Reserve roughly **6–10 GB VRAM** for this short, batch-one experiment (estimate,
  not a measured guarantee). Quantization does not compress every tensor.
- Download/startup may take several minutes; speed depends on Kaggle and the network.
- Cache goes to `/tmp/llada-hf-cache`, outside saved notebook outputs; it disappears
  when the VM is recycled. Small results go to `/kaggle/working/llada_results`.

The original LLaDA project recommends Transformers 4.38.2. This notebook pins that
API and its companion packages, preserves Kaggle's installed PyTorch, and checks
the actual CUDA/4-bit kernels. It does not install FlashAttention, xformers, or vLLM.


## 1. Install the small inference stack

Run this first. The PyTorch constraint prevents pip from silently replacing Kaggle's
GPU runtime. Unrelated preinstalled packages can report dependency conflicts because
this is an older Transformers stack; use a dedicated notebook and do not import those
packages here. A failed pip exit or a failed check below is a real blocker.


In [1]:
import sys, subprocess, importlib.metadata as metadata
from pathlib import Path

if not (3, 10) <= sys.version_info[:2] <= (3, 12):
    raise RuntimeError('This pinned setup targets Python 3.10–3.12. Use a compatible Kaggle environment.')

PINS = {
    'transformers': '4.38.2',
    'tokenizers': '0.15.2',
    'accelerate': '0.27.2',
    'bitsandbytes': '0.45.5',
    'huggingface-hub': '0.30.2',
    'safetensors': '0.5.3',
}
def installed(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None

torch_version = installed('torch')
if torch_version is None:
    raise RuntimeError('PyTorch is missing. Start a Kaggle GPU notebook environment.')
constraint = Path('/tmp/llada-torch-constraint.txt')
constraint.write_text(f'torch=={torch_version}\n')
needs_install = any(installed(k) != v for k, v in PINS.items())
already_imported = any(k in sys.modules for k in
                       ['transformers', 'tokenizers', 'accelerate', 'bitsandbytes', 'huggingface_hub', 'safetensors'])
if needs_install:
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
        '--no-cache-dir', '-c', str(constraint),
        *[f'{k}=={v}' for k, v in PINS.items()],
    ])
    if already_imported:
        raise RuntimeError('Installation completed. Restart the kernel/session now, then rerun from the top.')
print('Package pins ready. Kaggle PyTorch preserved:', torch_version)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.7/130.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 119.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 293.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.0/280.0 kB 375.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 271.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 481.4/481.4 kB 379.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.6/471.6 kB 244.7 MB/s eta 0:00:00
  Attempting uninstall: safetensors
    Found existing installation: safetensors 0.7.0
    Uninstalling safetensors-0.7.0:
      Successfully uninstalled safetensors-0.7.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.11.0
    Uninstalling huggingface_hub-1.11.0:
      Successfully uninstalled huggingface_hub-1.11.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizer

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.37.1 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 0.30.2 which is incompatible.
sentence-transformers 5.4.1 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.38.2 which is incompatible.
gradio 5.50.0 requires huggingface-hub<2.0,>=0.33.5, but you have huggingface-hub 0.30.2 which is incompatible.


## 2. Check the GPU and quantization kernels before downloading 8B weights

We use FP16 computation on T4/P100. BF16 support must not be assumed on those GPUs.
The CUDA version relevant to bitsandbytes is `torch.version.cuda`, which can differ
from the maximum version displayed by `nvidia-smi`.


In [2]:
import os, json, platform, shutil, gc, time
os.environ['HF_HOME'] = '/tmp/llada-hf-cache'
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '120'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import torch
import transformers, accelerate, bitsandbytes as bnb

for package, expected in PINS.items():
    if metadata.version(package) != expected:
        raise RuntimeError(f'{package}: expected {expected}. Rerun installation in a fresh kernel.')
if transformers.__version__ != PINS['transformers']:
    raise RuntimeError('Old imported Transformers is still in memory. Restart the kernel.')
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU. Enable a GPU accelerator and restart the session.')

DEVICE = torch.device('cuda:0')
DTYPE = torch.float16
print('Python:', platform.python_version(), '| Torch:', torch.__version__, '| CUDA:', torch.version.cuda)
for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f'GPU {i}: {torch.cuda.get_device_name(i)} | free {free/2**30:.1f}/{total/2**30:.1f} GiB')
cuda_version = tuple(map(int, torch.version.cuda.split('.')[:2]))
if not (11, 7) <= cuda_version <= (12, 8):
    raise RuntimeError('This bitsandbytes pin targets CUDA 11.7–12.8. See the runtime recovery section at the end.')
if torch.cuda.get_device_capability(0) < (6, 0):
    raise RuntimeError('Use T4, P100, or a newer GPU for this notebook.')
if torch.cuda.mem_get_info(0)[0] < 10 * 2**30:
    raise RuntimeError('Less than 10 GiB free on GPU 0. Stop other models or start a fresh session.')
Path(os.environ['HF_HOME']).mkdir(parents=True, exist_ok=True)
print('Cache disk free:', round(shutil.disk_usage(os.environ['HF_HOME']).free/2**30, 1), 'GiB')

# A small real NF4 forward pass catches CUDA/BNB incompatibility early.
with torch.inference_mode():
    layer = bnb.nn.Linear4bit(64, 64, bias=False, compute_dtype=DTYPE,
                             quant_type='nf4', compress_statistics=True).to(DEVICE)
    sample = torch.randn(2, 64, device=DEVICE, dtype=DTYPE)
    probe = layer(sample)
    assert probe.shape == (2, 64) and torch.isfinite(probe).all(), 'NF4 kernel produced invalid output.'
torch.cuda.synchronize()
del layer, sample, probe
gc.collect()
torch.cuda.empty_cache()
print('GPU + FP16 + NF4 test passed.')


Python: 3.12.13 | Torch: 2.10.0+cu128 | CUDA: 12.8
GPU 0: Tesla T4 | free 14.5/14.6 GiB
GPU 1: Tesla T4 | free 14.5/14.6 GiB
Cache disk free: 1118.8 GiB
GPU + FP16 + NF4 test passed.


## 3. Resolve a fixed checkpoint revision and check download size

`trust_remote_code=True` is needed because LLaDA supplies its own Python architecture.
It executes code from the model repository. The revision below points at the inspected
official model history rather than following a moving `main` branch. We resolve and
record its complete commit hash, then use that same hash for model and tokenizer.

Leave `USE_HF_SECRET=False` initially. If authenticated access is needed, create a
Kaggle secret named `HF_TOKEN` in Add-ons → Secrets, enable it for this notebook,
and set the switch to True. Never paste a token into a public notebook.


In [3]:
from huggingface_hub import HfApi

MODEL_ID = 'GSAI-ML/LLaDA-8B-Instruct'
REVISION = '08b83a6'
USE_HF_SECRET = False
HF_TOKEN = None
if USE_HF_SECRET:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')

info = HfApi(token=HF_TOKEN).model_info(MODEL_ID, revision=REVISION, files_metadata=True)
RESOLVED_REVISION = info.sha
weight_bytes = sum((f.size or 0) for f in info.siblings if f.rfilename.endswith('.safetensors'))
cache_root = Path(os.environ['HF_HOME'])
cache_bytes = sum(p.stat().st_size for p in cache_root.rglob('*') if p.is_file() and not p.is_symlink())
missing_estimate = max(0, weight_bytes - cache_bytes)
free_disk = shutil.disk_usage(cache_root).free
print('Checkpoint commit:', RESOLVED_REVISION)
print('Original safetensors download:', round(weight_bytes/2**30, 2), 'GiB')
if free_disk < missing_estimate + 3 * 2**30:
    raise RuntimeError('Insufficient cache disk. Free space or start a fresh session before loading.')
print('Download preflight passed. Public access works.')


Checkpoint commit: 08b83a6feb34df1a6011b80c3c00c7563e963b07
Original safetensors download: 14.93 GiB
Download preflight passed. Public access works.


## 4. Load the tokenizer and 4-bit model

Use `AutoModel` and the custom sampler below. Do not substitute a normal text-generation
pipeline or standard autoregressive `model.generate()`.
The whole quantized model is explicitly placed on GPU 0; no CPU/disk offload or
multi-GPU sharding is involved. Do not call `.cuda()`, `.half()`, or `.to(...)` on the
loaded model. It was already placed and quantized by the loader.


In [18]:
from transformers import AutoTokenizer, AutoModel, BitsAndBytesConfig

if 'model' in globals():
    raise RuntimeError('A model is already loaded. Skip this cell, or restart before loading another copy.')

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, revision=RESOLVED_REVISION, token=HF_TOKEN, trust_remote_code=True)
MASK_ID = 126336  # specific to this checkpoint; mask_token_id is not configured
assert tokenizer.convert_tokens_to_ids('<|mdm_mask|>') == MASK_ID, 'Unexpected tokenizer revision.'
quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=DTYPE,
)
started = time.perf_counter()
model = AutoModel.from_pretrained(
    MODEL_ID, revision=RESOLVED_REVISION, token=HF_TOKEN,
    trust_remote_code=True, use_safetensors=True,
    torch_dtype=DTYPE, quantization_config=quantization,
    device_map={'': 0}, low_cpu_mem_usage=True,
).eval()
model.config.use_cache = False
assert getattr(model, 'is_loaded_in_4bit', False), 'Quantized loading did not take effect.'
print('Loaded in', round(time.perf_counter() - started, 1), 'seconds')
print('Model footprint:', round(model.get_memory_footprint()/2**30, 2), 'GiB')
print('Quantized linear layers:', sum(isinstance(m, bnb.nn.Linear4bit) for m in model.modules()))
print('Device map:', model.hf_device_map)
print('Mask token:', repr(tokenizer.convert_ids_to_tokens(MASK_ID)), MASK_ID)


RuntimeError: A model is already loaded. Skip this cell, or restart before loading another copy.

## 5. What the sampler does

An autoregressive model predicts the next token, appends it, and repeats. Here we
append a fixed number of `[MASK]` positions. Each forward pass predicts tokens at
all positions using bidirectional context. We keep a quota of the most confident
predictions, then predict again with the remaining masks.

This teaching implementation supports **one unpadded prompt**, deterministic token
selection, confidence-based filling, and optional blocks. It omits classifier-free
guidance and stochastic sampling. It is independently written around the method in
the [official sampler](https://github.com/ML-GSAI/LLaDA/blob/main/generate.py).
It also excludes the mask token from predictions to guarantee progress, and computes
ranking probabilities in FP32 for stability. These are implementation differences;
this notebook is not a reproduction of the paper's reported scores.

| Parameter | Meaning | First run |
|---|---|---|
| `gen_length` | Number of answer token slots (tokens are not words) | 32 |
| `steps` | Total forward passes across all blocks | 16 |
| `block_length` | Slots filled together before moving to next block | 32 |
| `trace_every` | Save a partial answer every N forward passes | 4 |

`gen_length` must divide evenly into blocks. `steps` must divide evenly across the
number of blocks, with between 1 and `block_length` steps per block. More steps
usually trade speed for quality, but do not guarantee a better answer. Default
temperature is effectively zero; rerunning the same prompt is not a diversity test.


In [5]:
def transfer_schedule(gen_length, block_length, steps):
    values = (gen_length, block_length, steps)
    if any(type(v) is not int or v < 1 for v in values):
        raise ValueError('Lengths and steps must be positive integers.')
    if gen_length % block_length:
        raise ValueError('gen_length must be divisible by block_length.')
    blocks = gen_length // block_length
    if steps % blocks:
        raise ValueError('steps must be divisible by the number of blocks.')
    per_block = steps // blocks
    if not 1 <= per_block <= block_length:
        raise ValueError('Use between one and block_length steps per block.')
    base, remainder = divmod(block_length, per_block)
    return [base + (i < remainder) for i in range(per_block)]


@torch.inference_mode()
def denoise(model, prompt_ids, *, mask_id=126336, gen_length=32,
            steps=16, block_length=32, trace_every=0):
    quota = transfer_schedule(gen_length, block_length, steps)
    if prompt_ids.ndim != 2 or prompt_ids.shape[0] != 1 or prompt_ids.shape[1] == 0:
        raise ValueError('Supply one nonempty, unpadded prompt with shape [1, tokens].')
    if (prompt_ids == mask_id).any():
        raise ValueError('The prompt may not contain the special mask token.')
    if not isinstance(trace_every, int) or trace_every < 0:
        raise ValueError('trace_every must be a nonnegative integer.')
    prefix_len = prompt_ids.shape[1]
    tokens = torch.cat((prompt_ids, torch.full(
        (1, gen_length), mask_id, device=prompt_ids.device, dtype=torch.long)), dim=1)
    history = []
    if trace_every:
        history.append({'step': 0, 'remaining_masks': gen_length,
                        'token_ids': tokens[0, prefix_len:].cpu().tolist()})
    step = 0
    for start in range(prefix_len, prefix_len + gen_length, block_length):
        end = start + block_length
        for count in quota:
            # Bidirectional prediction: no causal mask and no KV cache.
            output = model(input_ids=tokens, use_cache=False)
            # Only score the current block; discard full-sequence logits promptly.
            scores = output.logits[0, start:end].float().clone()
            del output
            if not torch.isfinite(scores).all():
                raise FloatingPointError('Non-finite logits. Restart and verify FP16/NF4 kernels.')
            scores[:, mask_id] = -torch.inf
            chosen = scores.argmax(dim=-1)
            # log probability of the chosen token; same ranking as probability.
            confidence = scores.gather(1, chosen[:, None]).squeeze(1) - scores.logsumexp(dim=-1)
            eligible = tokens[0, start:end].eq(mask_id)
            confidence = confidence.masked_fill(~eligible, -torch.inf)
            positions = confidence.topk(count).indices
            tokens[0, start + positions] = chosen[positions]
            step += 1
            if trace_every and (step % trace_every == 0 or step == steps):
                history.append({'step': step,
                                'remaining_masks': int(tokens[0, prefix_len:].eq(mask_id).sum()),
                                'token_ids': tokens[0, prefix_len:].cpu().tolist()})
            del scores, chosen, confidence
    assert torch.equal(tokens[:, :prefix_len], prompt_ids), 'Prompt was modified.'
    assert not tokens[:, prefix_len:].eq(mask_id).any(), 'Unfilled output positions.'
    return tokens[:, prefix_len:], history


## 6. Test sampler invariants without the real model

This tiny CPU test uses synthetic logits. It checks prompt preservation, filling all
slots, block boundaries, an uneven quota, and parameter rejection. It checks mechanics,
not language quality or the real checkpoint's numerical behavior.


In [6]:
from types import SimpleNamespace

class ToyPredictor:
    def __init__(self): self.seen = []
    def __call__(self, input_ids, use_cache=False):
        self.seen.append(input_ids.clone())
        logits = torch.zeros(*input_ids.shape, 12)
        logits[..., 3] = 8.0
        return SimpleNamespace(logits=logits)

toy = ToyPredictor()
toy_prompt = torch.tensor([[1, 2]])
toy_out, toy_trace = denoise(toy, toy_prompt, mask_id=11, gen_length=8,
                           block_length=4, steps=4, trace_every=1)
assert toy_out.tolist() == [[3] * 8]
assert [row['remaining_masks'] for row in toy_trace] == [8, 6, 4, 2, 0]
assert all(torch.equal(x[:, :2], toy_prompt) for x in toy.seen)
assert all(x[:, 6:].eq(11).all() for x in toy.seen[:2])
assert sum(transfer_schedule(10, 10, 3)) == 10
for args in [(7, 4, 4), (8, 4, 3), (8, 4, 10), (0, 4, 2)]:
    try:
        transfer_schedule(*args)
    except ValueError:
        pass
    else:
        raise AssertionError(f'Invalid settings accepted: {args}')
del toy, toy_out, toy_trace
print('Sampler invariant tests passed.')


Sampler invariant tests passed.


## 7. A convenient chat function

Use the checkpoint's chat template, then tokenize with `add_special_tokens=False`
to avoid duplicating template tokens. We keep a conservative **512-token total**
limit for this tutorial; it is not the model's maximum context length.

Generation fills the whole requested canvas. For display, we trim at the first
end-of-message/end-of-text token **after denoising is complete**. Do not stop a
partly masked sequence at an early EOS prediction: later positions are still unresolved.


In [7]:
RUNS = []
OUTPUT_DIR = Path('/kaggle/working/llada_results')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def clean_answer(token_ids):
    stop_ids = {126081, 126348}  # EOS/EOT in original LLaDA sampler
    if isinstance(tokenizer.eos_token_id, int):
        stop_ids.add(tokenizer.eos_token_id)
    end = next((i for i, t in enumerate(token_ids) if t in stop_ids), len(token_ids))
    return tokenizer.decode(token_ids[:end], skip_special_tokens=True).strip()

def ask(prompt, *, gen_length=32, steps=16, block_length=32, trace_every=0):
    transfer_schedule(gen_length, block_length, steps)
    if not isinstance(prompt, str) or not prompt.strip():
        raise ValueError('Enter a nonempty text prompt.')
    text = tokenizer.apply_chat_template(
        [{'role': 'user', 'content': prompt}], tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, add_special_tokens=False, return_tensors='pt')['input_ids'].to(DEVICE)
    if inputs.shape[1] + gen_length > 512:
        raise ValueError('Tutorial limit: prompt tokens + gen_length must be <=512. Shorten the prompt.')
    torch.manual_seed(42)
    torch.cuda.manual_seed_all(42)
    torch.cuda.reset_peak_memory_stats(0)
    torch.cuda.synchronize()
    start = time.perf_counter()
    try:
        generated, history = denoise(model, inputs, mask_id=MASK_ID, gen_length=gen_length,
                                    steps=steps, block_length=block_length, trace_every=trace_every)
        torch.cuda.synchronize()
    except torch.cuda.OutOfMemoryError:
        print('OOM: restart if necessary, then use a shorter prompt and gen_length=16, block_length=16, steps=8.')
        raise
    elapsed = time.perf_counter() - start
    ids = generated[0].cpu().tolist()
    record = {
        'prompt': prompt, 'answer': clean_answer(ids), 'generated_token_ids': ids,
        'prompt_tokens': int(inputs.shape[1]), 'gen_length': gen_length, 'steps': steps,
        'block_length': block_length, 'seconds': elapsed,
        'peak_allocated_gib': torch.cuda.max_memory_allocated(0)/2**30,
        'trace': history,
    }
    RUNS.append(record)
    print(record['answer'] or '[Empty displayed answer; inspect raw token IDs / trace.]')
    print(f"\n{elapsed:.1f}s | {steps} forward passes | peak allocated {record['peak_allocated_gib']:.2f} GiB")
    return record


In [8]:
first = ask('Explain what a painting is in one short sentence.', trace_every=4)
assert len(first['generated_token_ids']) == 32
assert MASK_ID not in first['generated_token_ids']
print('Real-model smoke test passed: output length is correct and no masks remain.')


A painting is a visual art created by applying pigments or dyes to a surface, typically a canvas or other medium, to form an image or design.

19.1s | 16 forward passes | peak allocated 5.40 GiB
Real-model smoke test passed: output length is correct and no masks remain.


### Watch the masks disappear

At the start, every answer position is masked. Each row below shows the retained
tokens after more forward passes. Special tokens are intentionally visible here.
Filling positions out of order is the feature to notice. The smoke test does not
assert an exact sentence because quantization/hardware can affect predictions.


In [9]:
for row in first['trace']:
    print(f"Step {row['step']:2d} | masks left: {row['remaining_masks']:2d}")
    print(tokenizer.decode(row['token_ids'], skip_special_tokens=False))
    print()


Step  0 | masks left: 32
<|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|>

Step  4 | masks left: 24
A painting is a visual<|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|>.<|eot_id|><|endoftext|>

Step  8 | masks left: 16
A painting is a visual art created by applying pigments<|mdm_mask|><|mdm_mask|><|mdm_mask|> a<|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|> t

## 8. Try your own prompt

Edit `MY_PROMPT` and rerun. For a quality-oriented setting, try 64 slots and 64
steps. Reducing steps mainly reduces time; reducing sequence length or batch size
is what substantially reduces activation memory. Four-bit inference may change quality.


In [16]:
MY_PROMPT = 'what is the capital of iran?'
answer = ask(MY_PROMPT, gen_length=64, steps=32, block_length=32)


The capital of Iran is Tehran.

10.7s | 32 forward passes | peak allocated 5.40 GiB


## 9. Bridge toward ArtELingo: text-conditioned practice only

**LLaDA-8B-Instruct here cannot see an image.** The next example supplies a made-up
visual description and an emotion as text. It teaches prompting, but it is not image
captioning, a dataset example, or a valid ArtELingo result.


In [19]:
visual_description = 'A small boat rests on a quiet lake under a pale blue morning sky.'
emotion = 'sad'
art_prompt = (
    f'Visual description: {visual_description}\n'
    f'Emotion to express: {emotion}\n'
    'Write one short English sentence explaining why this scene could evoke that emotion. '
    'Use only the supplied visual details.'
)
art_example = ask(art_prompt, gen_length=64, steps=32, block_length=32)


The small boat resting alone on the quiet lake under a pale blue morning sky may evoke sadness, as the stillness of the lake and the muted blue of the sky suggest a sense of isolation and melancholy.

11.1s | 32 forward passes | peak allocated 5.41 GiB


## 10. Save the run and environment

Download the JSON files from Kaggle's output browser after running this cell, or use
Save Version to preserve them. These contain your prompts, so review before publishing.
Weights and tokens are not exported. The manifest records the complete checkpoint
revision, core package versions, GPU, sampler choices, and quantization settings.


In [ ]:
manifest = {
    'model': MODEL_ID, 'revision': RESOLVED_REVISION,
    'python': platform.python_version(), 'torch': torch.__version__, 'cuda': torch.version.cuda,
    'packages': {k: metadata.version(k) for k in PINS},
    'gpu': torch.cuda.get_device_name(0), 'device': str(DEVICE),
    'quantization': 'NF4 double quantization; FP16 compute',
    'sampler': 'tutorial v1; greedy confidence filling; mask token suppressed; batch=1; CFG=0',
    'seed': 42,
}
(OUTPUT_DIR / 'environment.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
(OUTPUT_DIR / 'generations.json').write_text(json.dumps(RUNS, ensure_ascii=False, indent=2), encoding='utf-8')
print('Saved:', OUTPUT_DIR / 'environment.json')
print('Saved:', OUTPUT_DIR / 'generations.json')


## Troubleshooting: fix the earliest failing cell

| Symptom | Likely reason / next action |
|---|---|
| No CUDA GPU | Choose GPU accelerator, check quota/account verification, restart session. |
| Installer says restart / import or tokenizer error | Restart kernel after changing installed packages; rerun from the first cell. Do not mix random upgrades into this stack. |
| DNS error / timeout | Enable Internet; retry the failed download cell. Existing cache files are normally reused. |
| 401/403/429 | Check checkpoint URL, network restrictions or rate limit; optionally use your own HF_TOKEN secret. Never print it. |
| `triton.ops` missing / BNB CUDA setup failure | Check installed pins and restart. Run the tiny NF4 test before retrying model loading. See runtime recovery below. |
| `no kernel image`, unsupported CUDA | GPU architecture and the installed PyTorch/BNB binary may disagree. Use the recovery procedure below. |
| BF16/FlashAttention error | Use the provided FP16 configuration. Do not add `flash_attention_2`. |
| `.to` unsupported for 4-bit / device mismatch | Keep Accelerate pin and `device_map={'': 0}`; do not move the loaded model afterward. |
| CUDA out of memory | Clear other models/restart; batch remains 1; shorten prompt; use 16 slots, block length 16, 8 steps. Lower steps alone is not a memory fix. |
| CPU RAM crashes on load | Use `low_cpu_mem_usage=True`, load one copy only, and start a fresh GPU session. |
| Disk full | Original weights still need disk space. Cache is separate from output; check `/tmp` space. |
| Assertion about KV cache | Use the custom sampler and `use_cache=False`, not autoregressive `.generate()`. |
| Length/steps ValueError | For 64 slots and blocks of 32 there are 2 blocks; steps must be even and at most 64. |
| Empty, repeated, or truncated text | Inspect raw trace; use the chat template; try 64 slots / 64 steps; simplify the request. Quantization can affect quality. |
| Second GPU idle | Expected. This notebook deliberately places one quantized model on GPU 0. |

### Runtime recovery (only if the CUDA/NF4 test fails)

Kaggle's base environment can change. The pinned inference packages do not pin its
PyTorch build. For a T4 or P100, a fallback is the following matched CUDA 12.1 stack
from the official PyTorch index. In a **fresh dedicated notebook**, run this as a
separate cell, **restart the kernel**, and then rerun this notebook from cell 1:

```python
import subprocess, sys
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '--no-cache-dir',
    'torch==2.5.1', 'torchvision==0.20.1', 'torchaudio==2.5.1',
    '--index-url', 'https://download.pytorch.org/whl/cu121',
])
```

This downloads several GB and replaces the session's PyTorch stack; it may conflict
with other Kaggle packages, which is why it is a dedicated-notebook fallback.
It targets Python 3.10–3.12 and T4/P100, not newer GPU architectures requiring a newer
CUDA runtime. You do not need to install the system CUDA toolkit or change the driver.
If the tiny test still fails, keep the complete first traceback and the environment
printout; do not continue downloading weights or keep layering package upgrades.

### Optional cleanup

To release model memory while keeping your session, run:
```python
del model
gc.collect()
torch.cuda.empty_cache()
```
Restarting is more reliable after an OOM traceback retains tensors. Stop the Kaggle
session when finished so it no longer consumes GPU quota.


## Later: turning this into ArtELingo research

ArtELingo contains artwork-related emotion explanations across languages, rather
than only literal object descriptions. Its original release emphasizes English,
Arabic and Chinese. Decide whether the emotion is provided at inference time or
must be inferred; ground-truth emotion is privileged input unless that is your task.

Two possible research routes:
1. **Pipeline baseline:** image → vision model's description → this dLLM's emotion
   explanation. This is a two-stage system; image errors propagate to the language model.
2. **Image-conditioned dLLM:** vision encoder → learned projector/image embeddings →
   diffusion language model. It requires multimodal training or a checkpoint designed
   for it, such as the separate LLaDA-V project. Its environment and memory requirements
   need a separate notebook; do not assume this loader can load it.

For research, use the dataset's official splits and verify artwork IDs do not leak
across your chosen train/validation/test sets. Keep all captions for the same artwork
together when constructing any new split. Report results per language, distinguish
emotion-conditioned from image-only inputs, and compare to an autoregressive baseline
under the same visual input and data protocol. Do not feed reference captions into
an image-only test prompt. Evaluate visual grounding and emotion relevance alongside
text similarity; do not assume Arabic performance from an English smoke test.

Four-bit loading here is an inference convenience, not a finetuning recipe. Diffusion
training requires a masking/noise objective; ordinary next-token SFT is not a drop-in
replacement. The custom LLaDA wrapper does not calculate a training loss from `labels`.

### Primary references (consulted 2026-10-01)
- [Official LLaDA code and loading instructions](https://github.com/ML-GSAI/LLaDA)
- [LLaDA checkpoint](https://huggingface.co/GSAI-ML/LLaDA-8B-Instruct)
- [Model revision history](https://huggingface.co/GSAI-ML/LLaDA-8B-Instruct/commits/main)
- [Official denoising implementation](https://github.com/ML-GSAI/LLaDA/blob/main/generate.py)
- [bitsandbytes 0.45.5 installation support](https://github.com/bitsandbytes-foundation/bitsandbytes/blob/0.45.5/docs/source/installation.mdx)
- [PyTorch previous versions / CUDA 12.1 fallback](https://pytorch.org/get-started/previous-versions/)
- [Kaggle notebook setup](https://www.kaggle.com/docs/notebooks)
- [ArtELingo project](https://www.artelingo.org/artelingo-emnlp22)
- [ArtELingo data and baseline code](https://github.com/Vision-CAIR/artelingo)
